* **概要**: 1990年代のインターネット掲示板（ニュースグループ）の投稿を集めたデータセットです。
* **データ数**: 約18,000件のテキスト文書。
これに対して、LDAモデルを適用してみる

In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

異なるカテゴリーの記事を4つ,そして関係ないものを取り除く
datesetの中身(Bunch)は、dataset.data:テキストの集合,dataset.target:カテゴリの数値ラベル,dataset.target_names:カテゴリの名前リスト見たいな感じ

In [ ]:
categories = ['alt.atheism', 'comp.graphics', 'sci.space', 'rec.sport.baseball']

# データの取得（ヘッダーやフッターなどのノイズを除去して純粋なテキストのみを取得）
print("Loading dataset...")
dataset = fetch_20newsgroups(subset='train', categories=categories, 
                             shuffle=True, random_state=42,
                             remove=('headers', 'footers', 'quotes'))
raw_texts = dataset.data
print(f"Loaded {len(raw_texts)} documents.")

Loading dataset...
Loaded 2254 documents.


In [12]:
raw_texts[0:3]

["\nA 68070 is just a 68010 with a built in MMU.  I don't even think that Moto.\nmanufactures them.\n\n                                  - Ian Romanick\n                                    Dancing Fool of Epsilon",
 "Hello, I realize that this might be a FAQ but I have to ask since I don't get a\nchange to read this newsgroup very often.  Anyways for my senior project I need\nto convert an AutoCad file to a TIFF file.  Please I don't need anyone telling\nme that the AutoCAD file is a vector file and the TIFF is a bit map since I\nhave heard that about 100 times already I would just like to know if anyone\nknows how to do this or at least point me to the right direction.",
 '\n\n\nHow do you know it\'s based on ignorance, couldn\'t that be wrong? Why would it\nbe wrong \nto fall into the trap that you mentioned? \n\nAlso, if I may, what the heck where we talking about and why didn\'t I keep \nsome comments on there to see what the line of thoughts were?\n\nMAC\n \n\n\n\n\n\n--\n********

countvectorizerで、
95%以上の文章に存在する単語は無視する、２個未満の単語は無視する。頻出頻度の高い単語(1000)に絞る,意味のない単語は消す
Xは、疎行列.もし普通に行列を作ると(d*V)の数になるので、メモリが破綻する。そのため、0の記録はしない。1の記録のみする
vocabは辞書こと。numpyの配列で渡されている

In [ ]:
vectorizer = CountVectorizer(max_df=0.95, min_df=2, 
                             max_features=1000, 
                             stop_words='english')


X = vectorizer.fit_transform(raw_texts)

# 辞書（IDから実際の単語文字列へのマッピング）を取得
vocab = vectorizer.get_feature_names_out()

In [18]:
print(type(X))

<class 'scipy.sparse._csr.csr_matrix'>


ここからLDAの具体的な内容について設定
トピック数は、4で決定
推定法はSVIを利用している。また、繰り返し学習数も10(ここでの10は全データを10回使うの意味、batch_size=128)にしている


In [19]:
n_topics = 4
lda = LatentDirichletAllocation(n_components=n_topics, 
                                max_iter=10, 
                                learning_method='online', 
                                random_state=42)

# 推定の実行（ここで裏側の期待値計算と足し算が高速で行われています）
lda.fit(X)


,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",4
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'online'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


 model.components_ は (K次元, V次元) の 行列のこと。
 topic.argsort()で、行列の値を小さい順に並べて、-10個、つまり大きい順に10個取り出して、インデックス番号を取り出す

In [44]:
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic #{topic_idx + 1}:")
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(" ".join(top_features))

print_top_words(lda, vocab, n_top_words=10)


Topic #1:
space nasa launch earth orbit 00 satellite shuttle lunar moon

Topic #2:
year good team game hit games think better runs players

Topic #3:
people don just think like god does know say time

Topic #4:
image edu graphics data available software file images use ftp


分類check,各文章に対して、トピック分布を獲得


In [46]:
topic_distribution = lda.transform(X)
predicted_topics = np.argmax(topic_distribution, axis=1)

In [49]:
predicted_topics[:10]

array([2, 3, 2, 0, 0, 2, 2, 2, 0, 3])

正答率は,ARIで判断(ランダムに記事を２個選んで、その２つが正しい別れ方をしていたら1点のようにする,まぐれで当たる分もあるのでそこはマイナスにしてある)

In [47]:
from sklearn.metrics import adjusted_rand_score
score = adjusted_rand_score(dataset.target, predicted_topics)
print(f"クラスタリング精度 (ARI): {score:.3f}")

クラスタリング精度 (ARI): 0.364
